In [4]:
"""
# llc + qemu RISC-V 中端测试

流程：SysY `.sy` -> 本项目中端生成 `.ll` -> LLVM `llc` 生成 RISC-V `.s` -> 交叉 GCC 静态链接 `runtime/sylib.c` -> `qemu-riscv64` 运行并对比 `.out`。

这个脚本用于在自研后端还不稳定时，固定 LLVM RISC-V 后端来观察中端优化在 RISC-V 目标上的正确性和相对性能趋势。
"""

# 中端 RISC-V 性能测试脚本：我们的编译器生成 .ll -> llc 生成 RISC-V .s -> 链接 sylib -> qemu-riscv64 运行 -> 对比 .out。
# 这个 notebook 刻意绕开当前不完善的自研后端，用 LLVM 官方 RISC-V 后端评估中端 IR 在 RISC-V 上的表现。

import csv
import hashlib
import os
import re
import shlex
import shutil
import subprocess
import time
from pathlib import Path
from difflib import unified_diff

class Color:
    GREEN = '\033[92m'
    RED = '\033[91m'
    YELLOW = '\033[93m'
    RESET = '\033[0m'

NOTEBOOK_CWD = Path.cwd().resolve()
if (NOTEBOOK_CWD / 'CMakeLists.txt').exists() and (NOTEBOOK_CWD / 'runtime').exists():
    PROJECT = NOTEBOOK_CWD
elif (NOTEBOOK_CWD / 'Compiler' / 'CMakeLists.txt').exists():
    PROJECT = NOTEBOOK_CWD / 'Compiler'
else:
    PROJECT = Path('/home/lyjy/Compiler/Compiler').resolve()

BUILD_DIR = PROJECT / 'build_middle'
COMPILER = BUILD_DIR / 'compiler_middle_end'
RUNTIME_C = PROJECT / 'runtime' / 'sylib.c'
#路径
DEFAULT_TEST_ROOT = PROJECT / 'testcases'

TEST_ROOT_ENV = os.environ.get('LLC_QEMU_TEST_ROOT')
TEST_ROOT = Path(TEST_ROOT_ENV).resolve() if TEST_ROOT_ENV else (DEFAULT_TEST_ROOT if DEFAULT_TEST_ROOT.exists() else PROJECT / 'testcases')

WORK_ROOT = PROJECT / 'temp_worker' / 'llc_qemu'
LL_ROOT = WORK_ROOT / 'll_out'
ASM_ROOT = WORK_ROOT / 'riscv_asm'
EXE_ROOT = WORK_ROOT / 'riscv_exe'
FAILURE_DIR = WORK_ROOT / 'failures'
REPORT_FILE = WORK_ROOT / 'failure_report.txt'
ASM_STATS_CSV = WORK_ROOT / 'asm_stats.csv'
SYLIB_OBJ = WORK_ROOT / 'sylib.o'

# 常用开关：日常全量测试直接运行即可；调单个 case 时改 SINGLE_CASES 或 MAX_CASES。
BUILD_BEFORE_TEST = True
QUIET_SETUP = True
CASE_PATTERNS = ['**/*.sy']
SINGLE_CASES_ENV = os.environ.get('LLC_QEMU_SINGLE_CASES')
SINGLE_CASES = [item.strip() for item in SINGLE_CASES_ENV.split(':') if item.strip()] if SINGLE_CASES_ENV else []
MAX_CASES_ENV = os.environ.get('LLC_QEMU_MAX_CASES')
MAX_CASES = int(MAX_CASES_ENV) if MAX_CASES_ENV else None
STOP_ON_FIRST_FAIL = False
COLLECT_ASM_STATS = True

COMPILE_TIMEOUT = 600
LLC_TIMEOUT = 120
LINK_TIMEOUT = 120
RUN_TIMEOUT = 300
BUILD_TIMEOUT = 900
CXX_COMPILER = os.environ.get('CXX', 'clang++-18')

LLC_OPT_LEVEL = os.environ.get('LLC_OPT_LEVEL', '2')
LLC_TRIPLE = os.environ.get('LLC_TRIPLE', 'riscv64-unknown-linux-gnu')
LLC_MATTR = os.environ.get('LLC_MATTR', '+m,+a,+f,+d,+c')
RISCV_MARCH = os.environ.get('RISCV_MARCH', 'rv64gc')
RISCV_MABI = os.environ.get('RISCV_MABI', 'lp64d')
LLC_TARGET_ABI = os.environ.get('LLC_TARGET_ABI', RISCV_MABI)
USE_OPAQUE_POINTERS = True
FAILURE_DETAIL_LIMIT = 120

if LLC_TARGET_ABI != RISCV_MABI:
    raise RuntimeError(f'LLC_TARGET_ABI({LLC_TARGET_ABI}) 必须和 RISCV_MABI({RISCV_MABI}) 一致，否则 llc 生成代码和 sylib.o 的浮点调用 ABI 会错位。')

results = {name: [] for name in ['AC', 'CE', 'LLC', 'LE', 'RE', 'WA', 'TLE', 'BAD']}
failure_details = []
asm_stats_rows = []
accepted_time = 0.0
judged_time = 0.0

def find_tool(env_name, candidates):
    env_value = os.environ.get(env_name)
    if env_value and shutil.which(env_value):
        return shutil.which(env_value)
    for name in candidates:
        path = shutil.which(name)
        if path:
            return path
    raise RuntimeError(f'未找到工具：{env_name}，候选={candidates}')

def detect_riscv_prefix():
    env_prefix = os.environ.get('RISCV_PREFIX')
    if env_prefix and shutil.which(env_prefix + 'gcc'):
        return env_prefix
    candidates = [
        'riscv64-linux-gnu-',
        'riscv64-unknown-linux-gnu-',
        '/home/r1c0/riscv/bin/riscv64-unknown-linux-gnu-',
    ]
    for prefix in candidates:
        if shutil.which(prefix + 'gcc'):
            return prefix
    raise RuntimeError('未找到 RISC-V 工具链，请安装 riscv64-linux-gnu-gcc 或设置 RISCV_PREFIX。')

LLC = find_tool('LLC', ['llc-14', 'llc'])
RISCV_PREFIX = detect_riscv_prefix()
RISCV_GCC = RISCV_PREFIX + 'gcc'
QEMU = find_tool('QEMU_RISCV64', ['qemu-riscv64'])

def format_filename(path, max_len=25):
    filename = Path(path).name
    if len(filename) > max_len:
        filename = filename[:max_len - 3] + '...'
    return filename.ljust(max_len)

def print_case(status, case, runtime_time, index, total):
    color = Color.GREEN if status == 'AC' else Color.YELLOW if status == 'TLE' else Color.RED
    gap = ' ' if len(status) >= 3 else '  '
    print(f'{color}[{status}]{Color.RESET}{gap}{format_filename(case)} | Runtime:{runtime_time:.6f}s | AC_Total:{accepted_time:>6.2f}s | [{index}/{total}]')

def run_command(args, timeout, stdin_path=None):
    stdin_file = stdin_path.open('rb') if stdin_path and stdin_path.exists() else None
    try:
        return subprocess.run(
            args,
            cwd=PROJECT,
            stdin=stdin_file,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
        )
    finally:
        if stdin_file:
            stdin_file.close()

def run_setup_command(args, stage, timeout):
    ret = subprocess.run(args, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=timeout)
    if ret.returncode == 0 and QUIET_SETUP:
        return ret
    if ret.returncode == 0:
        print(ret.stdout, end='')
        print(ret.stderr, end='')
        return ret
    print(f'{Color.RED}[{stage} failed]{Color.RESET} {shlex.join(args)}')
    if ret.stdout:
        print(ret.stdout[-3000:])
    if ret.stderr:
        print(ret.stderr[-3000:])
    raise RuntimeError(f'{stage} failed with exit code {ret.returncode}')

def build_middle_compiler():
    configure_cmd = [
        'cmake', '-B', str(BUILD_DIR), '-S', str(PROJECT),
        '-DCMAKE_BUILD_TYPE=Debug',
        '-DBUILD_MIDDLE_END=ON',
        '-DBUILD_FRONTEND_ONLY=OFF',
        '-DBUILD_FRONTEND_RELEASE=OFF',
        f'-DCMAKE_CXX_COMPILER={CXX_COMPILER}',
    ]
    build_cmd = ['cmake', '--build', str(BUILD_DIR), '--config', 'Debug', '--target', 'compiler_middle_end', '-j', '4']
    print(f'{Color.YELLOW}[SETUP]{Color.RESET} building middle-end compiler ...')
    run_setup_command(configure_cmd, 'configure', BUILD_TIMEOUT)
    run_setup_command(build_cmd, 'build', BUILD_TIMEOUT)
    print(f'{Color.YELLOW}[SETUP]{Color.RESET} compiler ready: {COMPILER}')

def prepare_workspace():
    shutil.rmtree(WORK_ROOT, ignore_errors=True)
    for path in [LL_ROOT, ASM_ROOT, EXE_ROOT, FAILURE_DIR]:
        path.mkdir(parents=True, exist_ok=True)
    runtime_cmd = [RISCV_GCC, '-O2', '-w', '-march=' + RISCV_MARCH, '-mabi=' + RISCV_MABI, '-c', str(RUNTIME_C), '-o', str(SYLIB_OBJ)]
    print(f'{Color.YELLOW}[SETUP]{Color.RESET} compiling sylib.c for RISC-V ...')
    run_setup_command(runtime_cmd, 'runtime', 120)

def resolve_case_path(path_text):
    raw = Path(path_text)
    if raw.is_absolute():
        return raw.resolve()
    candidate_from_project = (PROJECT / raw).resolve()
    if candidate_from_project.exists():
        return candidate_from_project
    candidate_from_test_root = (TEST_ROOT / raw).resolve()
    if candidate_from_test_root.exists():
        return candidate_from_test_root
    return candidate_from_project

def collect_cases():
    if SINGLE_CASES:
        cases = [resolve_case_path(p) for p in SINGLE_CASES]
    else:
        found = {case.resolve() for pattern in CASE_PATTERNS for case in TEST_ROOT.glob(pattern)}
        cases = sorted(case for case in found if case.suffix == '.sy')
    return cases[:MAX_CASES] if MAX_CASES is not None else cases

def case_key(case):
    try:
        rel = case.relative_to(TEST_ROOT)
    except ValueError:
        try:
            rel = case.relative_to(PROJECT)
        except ValueError:
            rel = Path(case.name)
    rel_text = rel.as_posix()
    digest = hashlib.md5(rel_text.encode('utf-8')).hexdigest()[:8]
    safe_rel = rel_text.replace('/', '__').replace(' ', '_')
    return f'{safe_rel}.{digest}'

def output_paths_for(case):
    key = case_key(case)
    return LL_ROOT / f'{key}.ll', ASM_ROOT / f'{key}.s', EXE_ROOT / f'{key}.riscv'

def command_detail(args, ret):
    stdout = ret.stdout.decode('utf-8', errors='replace')[-3000:]
    stderr = ret.stderr.decode('utf-8', errors='replace')[-3000:]
    return '\n'.join([
        'cmd: ' + shlex.join([str(a) for a in args]),
        f'returncode: {ret.returncode}',
        'stdout:',
        stdout,
        'stderr:',
        stderr,
    ])

def record_failure(status, case, detail):
    try:
        rel = str(case.relative_to(PROJECT))
    except ValueError:
        rel = str(case)
    failure_details.append((status, rel, detail.strip()))

def compile_to_ll(case, ll_file):
    cmd = [str(COMPILER), '-S', '-o', str(ll_file), str(case)]
    return run_command(cmd, COMPILE_TIMEOUT), cmd

def llc_to_asm(ll_file, asm_file):
    cmd = [LLC]
    if USE_OPAQUE_POINTERS:
        cmd.append('-opaque-pointers')
    cmd.extend([
        '-mtriple=' + LLC_TRIPLE,
        '-mattr=' + LLC_MATTR,
        '-target-abi=' + LLC_TARGET_ABI,
        '-O' + str(LLC_OPT_LEVEL),
        '-filetype=asm',
        str(ll_file),
        '-o', str(asm_file),
    ])
    return run_command(cmd, LLC_TIMEOUT), cmd

def link_asm(asm_file, exe_file):
    cmd = [
        RISCV_GCC,
        '-static',
        '-march=' + RISCV_MARCH,
        '-mabi=' + RISCV_MABI,
        str(asm_file),
        str(SYLIB_OBJ),
        '-o', str(exe_file),
    ]
    return run_command(cmd, LINK_TIMEOUT), cmd

def parse_runtime_seconds(stderr_bytes, fallback):
    stderr_text = stderr_bytes.decode('utf-8', errors='replace')
    match = re.search(r'TOTAL:\s*(\d+)H-(\d+)M-(\d+)S-(\d+)us', stderr_text)
    if not match:
        return fallback
    hours, minutes, seconds, microseconds = map(int, match.groups())
    return hours * 3600 + minutes * 60 + seconds + microseconds / 1_000_000

def run_and_compare(exe_file, input_file, expected_file):
    if not expected_file.exists():
        return 'BAD', 0.0, 'missing expected .out file'
    start = time.time()
    ret = run_command([QEMU, str(exe_file)], RUN_TIMEOUT, input_file)
    elapsed = time.time() - start
    runtime_time = parse_runtime_seconds(ret.stderr, elapsed)
    if ret.returncode < 0:
        return 'RE', runtime_time, command_detail([QEMU, str(exe_file)], ret)
    actual = ret.stdout.decode('utf-8', errors='replace')
    if actual and not actual.endswith('\n'):
        actual += '\n'
    actual += str(ret.returncode) + '\n'
    expected = expected_file.read_text(encoding='utf-8')
    diff = list(unified_diff(actual.splitlines(), expected.splitlines(), fromfile='actual', tofile='expected', lineterm=''))
    if diff:
        return 'WA', runtime_time, '\n'.join(diff[:FAILURE_DETAIL_LIMIT])
    return 'AC', runtime_time, ''

LOAD_OPS = {'lb', 'lh', 'lw', 'ld', 'lbu', 'lhu', 'lwu', 'flw', 'fld'}
STORE_OPS = {'sb', 'sh', 'sw', 'sd', 'fsw', 'fsd'}
BRANCH_OPS = {'beq', 'bne', 'blt', 'bge', 'bltu', 'bgeu', 'j', 'jr', 'jal', 'jalr', 'ret'}
MULDIV_OPS = {'mul', 'mulh', 'mulhu', 'mulhsu', 'mulw', 'div', 'divu', 'divw', 'divuw', 'rem', 'remu', 'remw', 'remuw'}
CALL_OPS = {'call', 'tail'}

def collect_asm_stats(case, asm_file):
    stats = {
        'case': str(case.relative_to(PROJECT)) if case.is_relative_to(PROJECT) else str(case),
        'instructions': 0,
        'loads': 0,
        'stores': 0,
        'branches': 0,
        'mul_div_rem': 0,
        'calls': 0,
    }
    for raw_line in asm_file.read_text(encoding='utf-8', errors='replace').splitlines():
        line = raw_line.split('#', 1)[0].strip()
        if not line or line.startswith('.') or line.endswith(':'):
            continue
        if ':' in line:
            line = line.split(':', 1)[1].strip()
            if not line or line.startswith('.'):
                continue
        op = line.split(None, 1)[0].rstrip(',')
        stats['instructions'] += 1
        stats['loads'] += op in LOAD_OPS
        stats['stores'] += op in STORE_OPS
        stats['branches'] += op in BRANCH_OPS
        stats['mul_div_rem'] += op in MULDIV_OPS
        stats['calls'] += op in CALL_OPS
    asm_stats_rows.append(stats)

def write_asm_stats():
    if not asm_stats_rows:
        return
    with ASM_STATS_CSV.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(asm_stats_rows[0].keys()))
        writer.writeheader()
        writer.writerows(asm_stats_rows)

def write_failure_report():
    if not failure_details:
        REPORT_FILE.write_text('No failures.\n', encoding='utf-8')
        return
    lines = []
    for status, rel, detail in failure_details:
        lines.append(f'[{status}] {rel}')
        if detail:
            lines.append(detail)
        lines.append('')
    REPORT_FILE.write_text('\n'.join(lines), encoding='utf-8')

def test_one_case(case, index, total):
    global accepted_time, judged_time
    ll_file, asm_file, exe_file = output_paths_for(case)
    input_file = case.with_suffix('.in')
    expected_file = case.with_suffix('.out')

    try:
        compile_ret, compile_cmd = compile_to_ll(case, ll_file)
    except subprocess.TimeoutExpired:
        results['TLE'].append(case)
        record_failure('TLE', case, f'compiler timeout after {COMPILE_TIMEOUT}s')
        print_case('TLE', case, 0.0, index, total)
        return 'TLE'
    if compile_ret.returncode != 0:
        results['CE'].append(case)
        record_failure('CE', case, command_detail(compile_cmd, compile_ret))
        print_case('CE', case, 0.0, index, total)
        return 'CE'

    try:
        llc_ret, llc_cmd = llc_to_asm(ll_file, asm_file)
    except subprocess.TimeoutExpired:
        results['TLE'].append(case)
        record_failure('TLE', case, f'llc timeout after {LLC_TIMEOUT}s')
        print_case('TLE', case, 0.0, index, total)
        return 'TLE'
    if llc_ret.returncode != 0:
        results['LLC'].append(case)
        record_failure('LLC', case, command_detail(llc_cmd, llc_ret))
        print_case('LLC', case, 0.0, index, total)
        return 'LLC'

    if COLLECT_ASM_STATS:
        collect_asm_stats(case, asm_file)

    try:
        link_ret, link_cmd = link_asm(asm_file, exe_file)
    except subprocess.TimeoutExpired:
        results['TLE'].append(case)
        record_failure('TLE', case, f'link timeout after {LINK_TIMEOUT}s')
        print_case('TLE', case, 0.0, index, total)
        return 'TLE'
    if link_ret.returncode != 0:
        results['LE'].append(case)
        record_failure('LE', case, command_detail(link_cmd, link_ret))
        print_case('LE', case, 0.0, index, total)
        return 'LE'

    try:
        status, runtime_time, detail = run_and_compare(exe_file, input_file, expected_file)
    except subprocess.TimeoutExpired:
        judged_time += RUN_TIMEOUT
        results['TLE'].append(case)
        record_failure('TLE', case, f'qemu timeout after {RUN_TIMEOUT}s')
        print_case('TLE', case, RUN_TIMEOUT, index, total)
        return 'TLE'

    judged_time += runtime_time
    if status == 'AC':
        accepted_time += runtime_time
    results[status].append(case)
    if status != 'AC':
        record_failure(status, case, detail)
    print_case(status, case, runtime_time, index, total)
    return status

def print_summary():
    write_failure_report()
    write_asm_stats()
    print()
    print('=' * 60)
    print(f'Compiler Error:  {len(results["CE"]):>3}')
    print(f'LLC Error:       {len(results["LLC"]):>3}')
    print(f'Linker Error:    {len(results["LE"]):>3}')
    print(f'Runtime Error:   {len(results["RE"]):>3}')
    print(f'Timeout Error:   {len(results["TLE"]):>3}')
    print(f'Wrong Answer:    {len(results["WA"]):>3}')
    print(f'Bad Test:        {len(results["BAD"]):>3}')
    print(f'{Color.GREEN}Accepted:        {len(results["AC"]):>3}{Color.RESET}')
    print('=' * 60)
    print(f'Accepted runtime: {accepted_time:.2f}s')
    print(f'Judged runtime:   {judged_time:.2f}s')
    print(f'Artifacts: {WORK_ROOT}')
    if COLLECT_ASM_STATS and asm_stats_rows:
        print(f'ASM stats: {ASM_STATS_CSV}')
    if failure_details:
        print(f'{Color.YELLOW}失败详情已写入:{Color.RESET} {REPORT_FILE}')
        for name in ['CE', 'LLC', 'LE', 'RE', 'WA', 'TLE', 'BAD']:
            if results[name]:
                print(f'{name}_LIST')
                for case in results[name]:
                    try:
                        print('  ' + str(case.relative_to(PROJECT)))
                    except ValueError:
                        print('  ' + str(case))

if BUILD_BEFORE_TEST:
    build_middle_compiler()
if not COMPILER.exists():
    raise FileNotFoundError(f'找不到中端编译器：{COMPILER}')
if not RUNTIME_C.exists():
    raise FileNotFoundError(f'找不到运行库：{RUNTIME_C}')
prepare_workspace()

test_list = collect_cases()
if not test_list:
    raise RuntimeError('没有找到任何 .sy 测试用例，请检查 TEST_ROOT/CASE_PATTERNS/SINGLE_CASES。')

print(f'{Color.YELLOW}[SETUP]{Color.RESET} project: {PROJECT}')
print(f'{Color.YELLOW}[SETUP]{Color.RESET} test root: {TEST_ROOT}')
print(f'{Color.YELLOW}[SETUP]{Color.RESET} llc: {LLC}, triple={LLC_TRIPLE}, abi={LLC_TARGET_ABI}, mattr={LLC_MATTR}, -O{LLC_OPT_LEVEL}')
print(f'{Color.YELLOW}[SETUP]{Color.RESET} gcc: {RISCV_GCC}, qemu: {QEMU}')
print(f'{Color.YELLOW}[SETUP]{Color.RESET} cases: {len(test_list)}, timeout: compile={COMPILE_TIMEOUT}s, llc={LLC_TIMEOUT}s, run={RUN_TIMEOUT}s')

for test_count, test in enumerate(test_list, 1):
    status = test_one_case(test, test_count, len(test_list))
    if status != 'AC' and STOP_ON_FIRST_FAIL:
        break

print_summary()

print('AC', len(results['AC']))
print('CE', len(results['CE']))
print('LLC', len(results['LLC']))
print('LE', len(results['LE']))
print('RE', len(results['RE']))
print('TLE', len(results['TLE']))
print('WA', len(results['WA']))
print('BAD', len(results['BAD']))
for name in ['CE', 'LLC', 'LE', 'RE', 'TLE', 'WA', 'BAD']:
    if results[name]:
        print(f'\n{name}_LIST')
        for case in results[name]:
            print(case)


[SETUP] building middle-end compiler ...
[SETUP] compiler ready: /home/lyjy/Compiler/Compiler/build_middle/compiler_middle_end
[SETUP] compiling sylib.c for RISC-V ...
[SETUP] project: /home/lyjy/Compiler/Compiler
[SETUP] test root: /home/lyjy/Compiler/Compiler/testcases
[SETUP] llc: /usr/bin/llc-14, triple=riscv64-unknown-linux-gnu, abi=lp64d, mattr=+m,+a,+f,+d,+c, -O2
[SETUP] gcc: riscv64-linux-gnu-gcc, qemu: /usr/bin/qemu-riscv64
[SETUP] cases: 382, timeout: compile=600s, llc=120s, run=300s
[AC]  2024-0ZE-13.sy            | Runtime:0.694420s | AC_Total:  0.69s | [1/382]
[AC]  2024-1WO-11.sy            | Runtime:12.124279s | AC_Total: 12.82s | [2/382]
[AC]  2024-2KM-21.sy            | Runtime:1.354430s | AC_Total: 14.17s | [3/382]
[AC]  2024-7UJ-12.sy            | Runtime:0.339863s | AC_Total: 14.51s | [4/382]
[AC]  2024-8NL-18.sy            | Runtime:0.661691s | AC_Total: 15.17s | [5/382]
[AC]  2024-931-6.sy             | Runtime:0.577337s | AC_Total: 15.75s | [6/382]
[AC]  2024-BAV